# NB24 — Phase A : calibration des taux de but `alpha` (moteur Poisson buteur)

À LANCER AVANT l'horizon des phases finales (`compute_robust_endgame_horizon`), une
fois (pré-tournoi), puis seulement si les cotes buteur changent. Calibre :
- `alpha` (buts/match intrinsèque) par buteur listé,
- `alpha_field` (taux des buteurs de FOND génériques, un par équipe),

pour que la fréquence simulée de *meilleur buteur* (convention dead-heat) colle aux
cotes buteur dé-viggées (Shin). Écrit le résultat dans la colonne `alpha` de
`data/CDM_2026_goal_scorer_and_favorite.csv` (buteurs + ligne `field`).

L'échelle de buts est ANCRÉE au réalisme (`g_fav` = buts attendus du favori du
marché, ~5) pour préserver la dynamique de *catch-up* Poisson (un buteur dont
l'équipe est éliminée voit son `M_restants` tomber à 0 -> total gelé).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy')
    !pip install shin numba
else:
    PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / 'data'

from mpp_project.core import calculate_true_outcome_probas_from_odds
from mpp_project.scorer_model import (
    simulate_team_matches_played, calibrate_scorer_alphas, load_scorer_players,
)

MARKET = DATA_DIR / 'CDM_2026_goal_scorer_and_favorite.csv'
ODDS = DATA_DIR / 'CDM_2026_group_stage_odds.csv'

# Paramètres de calibration
G_FAV = 5.0          # buts attendus du favori du marché (ancre de réalisme)
N_RUNS = 15000       # tournois simulés pour la distribution des matchs joués
BETA = 0.95          # amortissement force conditionnelle (cf. NB23)
N_POISSON = 3        # tirages Poisson par tournoi

In [ ]:
# 1. Cible = P(meilleur buteur) dé-viggée (Shin) sur les cotes buteur
df_market = pd.read_csv(MARKET)
df_odds = pd.read_csv(ODDS)
players, autre_gain, _ = load_scorer_players(df_market)
target = calculate_true_outcome_probas_from_odds(players['cote'].values.astype(float))
print(f'{len(players)} buteurs ({int(players.is_ghost.sum())} ghosts), cible Shin somme={target.sum():.3f}')

team_to_id = {str(t).strip().lower(): i for i, t in enumerate(df_odds['team'])}
nation_id = np.array([team_to_id[n] for n in players['nation']], dtype=np.int64)
field_nation_id = np.arange(len(df_odds), dtype=np.int64)  # un buteur de fond par équipe

In [ ]:
# 2. Distribution des matchs joués par équipe (poules + knockout)
matches, team_names = simulate_team_matches_played(df_odds, n_runs=N_RUNS, beta=BETA, seed=7)
print('E[matchs] top equipes :')
for i in np.argsort(-matches.mean(0))[:5]:
    print(f'  {team_names[i]:>12} {matches[:, i].mean():.2f}')

In [ ]:
# 3. Calibration (lien paramétrique E[buts]=c*target^p + champ de fond)
alpha, info = calibrate_scorer_alphas(
    target, nation_id, matches.astype(float), field_nation_id,
    g_fav=G_FAV, n_poisson_per_run=N_POISSON, verbose=True, seed=3,
)
print(f"\np={info['p']:.3f}  alpha_field={info['alpha_field']:.3f}  "
      f"MAE={info['mae']:.4f}  rankcorr={info['rankcorr']:.3f}  field_share={info['field_share']:.3f}")

In [ ]:
# 4. Contrôle : cible vs simulé + buts attendus
EM = matches[:, nation_id].mean(0)
ctrl = pd.DataFrame({
    'buteur': players['selection'], 'nation': players['nation'],
    'target': target, 'sim': info['sim_probs'],
    'alpha': alpha, 'E[buts]': alpha * EM,
}).sort_values('target', ascending=False)
ctrl.head(15).style.format({'target': '{:.3f}', 'sim': '{:.3f}', 'alpha': '{:.3f}', 'E[buts]': '{:.2f}'})

In [ ]:
# 5. Écriture dans le CSV (colonne alpha des scorers + ligne field)
sel_to_alpha = {players['selection'].iloc[i]: round(float(alpha[i]), 4) for i in range(len(players))}

def _set_alpha(r):
    cat = str(r['category']).strip().lower()
    sel = str(r['selection']).strip().lower()
    if cat == 'field':
        return round(float(info['alpha_field']), 4)
    if cat == 'scorer' and sel != 'autre':
        return sel_to_alpha.get(sel, '')
    return r['alpha'] if 'alpha' in r and pd.notna(r['alpha']) else ''

df_market['alpha'] = df_market.apply(_set_alpha, axis=1)
df_market.to_csv(MARKET, index=False)
print(f'alpha écrit dans {MARKET.name}. Prêt pour compute_robust_endgame_horizon.')